# Data Cleaning for TIGPS 2024 Student Data

This notebook performs a comprehensive data cleaning process on the file `TIGPS_W2_studentdata_ver0.csv`.

**Objective:** To prepare the dataset for analysis by addressing common data quality issues such as duplicates, missing values, and data inconsistencies.

**Steps:**
1.  **Load and Inspect Data:** Understand the structure and initial quality.
2.  **Duplicate Removal:** Identify and remove duplicate records.
3.  **Missing Value Analysis & Handling:** Identify missing data and apply appropriate strategies.
4.  **Data Type Conversion:** Ensure columns have the correct data types.
5.  **Save Cleaned Data:** Export the processed dataset.

## 1. Load and Inspect Data

First, we load the dataset and examine its basic properties: dimensions, column names, and data types.

In [1]:
import pandas as pd
import numpy as np
import os

# Define file paths
input_file = r"..\Data\2024data\TIGPS_W2_studentdata_ver0.csv"
output_file = r"..\Data\2024data\TIGPS_W2_studentdata_cleaned.csv"

# Load data
try:
    df = pd.read_csv(input_file)
    print("Data loaded successfully.")
    print(f"Shape: {df.shape}")
except FileNotFoundError:
    print(f"Error: File not found at {input_file}")

# Display first few rows
df.head()

Data loaded successfully.
Shape: (8892, 365)


,student_oid,student_id,qb_code,q_name,school_id,school_name,class,status,name,cell,...,v59_3h,v59_3m,v59_4h,v59_4m,v59_5,v60_1,v60_2,v61,v62,v63
0,s112011301024,s112011301024,CO01,2024TASAL-TIGPS國中學生問卷,11301,私立淡江高中附設國中,801,0,王譽文,0979205386,...,凌晨1點,30,早上5點,20,4. 非常好,0,4. 5-6杯,163.0,40.0,1
1,s112011301025,s112011301025,CO01,2024TASAL-TIGPS國中學生問卷,11301,私立淡江高中附設國中,801,0,吳律佑,0975295429,...,中午12點,30,早上8點,0,3. 還算好,1. 沒有,0,160.0,48.0,2
2,s112011301026,s112011301026,CO01,2024TASAL-TIGPS國中學生問卷,11301,私立淡江高中附設國中,801,0,宋胤安,0908606506,...,早上10點,0,早上8點,30,3. 還算好,0,0,172.0,60.0,4
3,s112011301028,s112011301028,CO01,2024TASAL-TIGPS國中學生問卷,11301,私立淡江高中附設國中,801,0,林松賢,0958861128,...,晚上11點,0,晚上9點,0,3. 還算好,1. 沒有,1. 沒有,170.0,65.0,5
4,s112011301029,s112011301029,CO01,2024TASAL-TIGPS國中學生問卷,11301,私立淡江高中附設國中,801,0,侯律遠,0958441313,...,凌晨4點,30,中午12點,45,2. 不太好,1. 沒有,0,-9.0,50.0,4


In [2]:
# Basic information about columns and data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8892 entries, 0 to 8891
Columns: 365 entries, student_oid to v63
dtypes: float64(2), int64(34), object(329)
memory usage: 24.8+ MB


## 2. Duplicate Removal

We check for duplicate rows and duplicate student IDs. Duplicate IDs might indicate data entry errors or multiple submissions.

In [3]:
# Check for full duplicate rows
duplicates = df.duplicated()
print(f"Number of duplicate rows: {duplicates.sum()}")

if duplicates.sum() > 0:
    df = df.drop_duplicates()
    print("Duplicate rows removed.")

# Check for duplicate student_oid (if it exists)
if 'student_oid' in df.columns:
    duplicate_ids = df.duplicated(subset=['student_oid'])
    print(f"Number of duplicate student_oid entries: {duplicate_ids.sum()}")
    # Strategy: Keep the first occurrence or investigate further. 
    # Here we will keep the first instance for simplicity in this automated flow.
    if duplicate_ids.sum() > 0:
        df = df.drop_duplicates(subset=['student_oid'], keep='first')
        print("Duplicate student_oid entries removed (kept first).")
else:
    print("'student_oid' column not found, skipping ID duplicate check.")

Number of duplicate rows: 0
Number of duplicate student_oid entries: 0


## 3. Missing Value Analysis & Handling

We identify columns with missing values. 

**Strategy:**
-   **High Missingness:** Columns with a very high percentage of missing values (e.g., > 90%) might be dropped if they aren't critical.
-   **Low Missingness:** For categorical columns, we might fill with a placeholder like "Unknown". For numerical, we might use mean/median or leave as is depending on analysis needs.
-   **Rows:** Rows with significant missing data might be removed.

In [4]:
# Calculate missing values per column
missing_counts = df.isnull().sum()
missing_percentage = (missing_counts / len(df)) * 100

# Filter to show only columns with missing values
missing_data = pd.DataFrame({'Missing Values': missing_counts, 'Percentage': missing_percentage})
missing_data = missing_data[missing_data['Missing Values'] > 0].sort_values(by='Percentage', ascending=False)

print("Columns with missing values:")
print(missing_data)

# Handling Missing Values
# 1. Drop columns with > 95% missing values (Conservative threshold)
cols_to_drop = missing_percentage[missing_percentage > 95].index
if len(cols_to_drop) > 0:
    print(f"Dropping columns with > 95% missing values: {list(cols_to_drop)}")
    df = df.drop(columns=cols_to_drop)

# 2. For remaining missing values, we can fill categorical with 'Unknown' and numerical with 0 or mean.
# This part is very context-dependent. For survey data, -9 usually represents missing/skip.
# We will check for common survey missing indicators like -9 or -6 and standardize if needed.
# But for true NaNs:
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna('Unknown')
    else:
        # For numeric columns in surveys, sometimes it's better to leave as NaN or set to a specific code like -99
        # Here we will imply no drastic imputation without user guidance, just reporting.
        pass

print("Filled missing categorical values with 'Unknown'.")

Columns with missing values:
       Missing Values  Percentage
kv2              8835   99.358974
kv16             8572   96.401260
email               7    0.078722
cell                2    0.022492
Dropping columns with > 95% missing values: ['kv2', 'kv16']
Filled missing categorical values with 'Unknown'.


## 4. Data Consistency & Cleaning

Specific checks for this dataset:
-   Ensure string columns don't have leading/trailing whitespace.
-   Standardize common values if needed.

In [5]:
# Strip whitespace from all string columns
df_obj = df.select_dtypes(['object'])
df[df_obj.columns] = df_obj.apply(lambda x: x.str.strip())
print("Whitespace stripped from string columns.")

# Check for unique values in key columns (e.g., status, gender if they exist)
if 'status' in df.columns:
    print("\nUnique values in 'status':")
    print(df['status'].unique())

if 'v1' in df.columns: # Assuming v1 might be gender based on some survey structures, or just checking first variable
    print("\nUnique values in 'v1':")
    print(df['v1'].unique())

Whitespace stripped from string columns.

Unique values in 'status':
[0]

Unique values in 'v1':
['2. 男' '1. 女']


## 5. Save Cleaned Data

Saving the cleaned dataframe to a new CSV file.

In [6]:
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Cleaned data saved to: {output_file}")

Cleaned data saved to: ..\Data\2024data\TIGPS_W2_studentdata_cleaned.csv
